In [1]:
import os
from tqdm import tqdm
import mysql.connector
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
host=os.getenv('host')
user=os.getenv('user')
password=os.getenv('password')
database=os.getenv('database')
mydb = mysql.connector.connect(
  host=host,
  user=user,
  password=password,
  database=database
)
mycursor = mydb.cursor()

In [8]:
query = """
select u.userid, u.favorite_genre, group_concat(distinct f.id separator ', ') as artist_id from user as u
inner join favorite as f on u.userid = f.user_id and f.type = "artist"
group by u.userid;
"""
mycursor.execute(query)

data = mycursor.fetchall()

print(data)

mydb.commit()
# df2 = pd.DataFrame(df1, columns=["usename", "password", "favorite_genre", "image", "email", "userid"])
# df2.head()


[(1, None, '0EmeFodog0BfCgMzAIvKQp, 3Nrfpe0tUJi4K4DXYWgMUX, 4VMYDCV2IEDYJArk749S6m, 66CXWjxzNUsdJxJ2JdwvnR, 6eUKZXaKkcviH0Ku9w2n3V, 6l3HvQ5sa6mXTsMTB19rO5, 6LuN9FCkKOj5PcnpouEgny'), (2, 'acoustic, afrobeat', '00FQb4jTyendYWaN8pK0wa, 0Y5tJX1MQlPlqiwlOH1tJY, 41MozSoPIsD1dJM0CLPjZF, 66CXWjxzNUsdJxJ2JdwvnR'), (3, '', '1VPmR4DJC1PlOtd0IADAO0, 246dkjvS1zLTtiykXe5h60, 2R21vXR83lH98kGeO99Y66, 4gzpq5DPGxSnKTe4SA8HAU, 4q3ewBCX7sLwd24euuV69X, 5K4W6rqBFWDnAN6FQUkS6x, 6l3HvQ5sa6mXTsMTB19rO5'), (4, 'tango, techno, trance', '1Cs0zKBU1kc0i8ypK3B9ai, 2R21vXR83lH98kGeO99Y66, 3MZsBdqDrRTJihTHQrO6Dq, 3TVXtAsR1Inumwj472S9r4, 4VMYDCV2IEDYJArk749S6m, 6l3HvQ5sa6mXTsMTB19rO5, 6M2wZ9GZgrQXHCFfjv46we, 7Ln80lUS6He07XvHI8qqHH'), (5, 'rock, pop, indie-pop, dance', '00FQb4jTyendYWaN8pK0wa, 0du5cEVh5yTK9QJze8zA0C, 0EmeFodog0BfCgMzAIvKQp, 0Y5tJX1MQlPlqiwlOH1tJY, 2YZyLoL8N0Wb9xBt1NhZWg, 6l3HvQ5sa6mXTsMTB19rO5, 790FomKkXshlbRYZFtlgla, 7Ln80lUS6He07XvHI8qqHH'), (6, 'jazz, blues, soul, funk', '0du5cEVh5yTK9QJze8zA0C, 0hCN

In [9]:
import pandas as pd


df1 = pd.DataFrame(data, columns=[
    "uid", "favorite_genre", "artist_id"
])
df1.head()

,uid,favorite_genre,artist_id
0,1,NaN,"0EmeFodog0BfCgMzAIvKQp, 3Nrfpe0tUJi4K4DXYWgMUX..."
1,2,"acoustic, afrobeat","00FQb4jTyendYWaN8pK0wa, 0Y5tJX1MQlPlqiwlOH1tJY..."
2,3,,"1VPmR4DJC1PlOtd0IADAO0, 246dkjvS1zLTtiykXe5h60..."
3,4,"tango, techno, trance","1Cs0zKBU1kc0i8ypK3B9ai, 2R21vXR83lH98kGeO99Y66..."
4,5,"rock, pop, indie-pop, dance","00FQb4jTyendYWaN8pK0wa, 0du5cEVh5yTK9QJze8zA0C..."


In [10]:
GENRES = [
    "acoustic","afrobeat","alt-rock","ambient","black-metal","blues","breakbeat",
    "cantopop","chicago-house","chill","classical","club","comedy","country",
    "dance","dancehall","death-metal","deep-house","detroit-techno","disco",
    "drum-and-bass","dub","dubstep","edm","electro","electronic","emo","folk",
    "forro","french","funk","garage","german","gospel","goth","grindcore",
    "groove","guitar","hard-rock","hardcore","hardstyle","heavy-metal","hip-hop",
    "house","indian","indie-pop","industrial","jazz","k-pop","metal","metalcore",
    "minimal-techno","new-age","opera","party","piano","pop","pop-film",
    "power-pop","progressive-house","psych-rock","punk","punk-rock","rock",
    "rock-n-roll","romance","sad","salsa","samba","sertanejo","show-tunes",
    "singer-songwriter","ska","sleep","songwriter","soul","spanish","swedish",
    "tango","techno","trance","trip-hop"
]

def build_user_genre_matrix(df1):
    # chỉ lấy 2 cột cần
    df = df1[["uid", "favorite_genre"]].copy()

    # xử lý null / rỗng
    df["favorite_genre"] = df["favorite_genre"].fillna("")
    
    # tách list genre
    df["favorite_genre"] = df["favorite_genre"].apply(
        lambda x: [g.strip() for g in x.split(",") if g.strip()]
    )

    # explode
    df = df.explode("favorite_genre")

    # tạo one-hot
    df = pd.get_dummies(df, columns=["favorite_genre"])

    # group lại theo user
    df = df.groupby("uid").max().reset_index()

    # đảm bảo đủ tất cả genre
    for g in GENRES:
        col = f"favorite_genre_{g}"
        if col not in df.columns:
            df[col] = 0

    # rename cho gọn
    df.columns = [c.replace("favorite_genre_", "") for c in df.columns]

    # sắp xếp cột
    df = df[["uid"] + GENRES]

    return df


df2 = build_user_genre_matrix(df1)
df2 = df2.astype(int)
print(df2.head())

   uid  acoustic  afrobeat  alt-rock  ambient  black-metal  blues  breakbeat  \
0    1         0         0         0        0            0      0          0   
1    2         1         1         0        0            0      0          0   
2    3         0         0         0        0            0      0          0   
3    4         0         0         0        0            0      0          0   
4    5         0         0         0        0            0      0          0   

   cantopop  chicago-house  ...  ska  sleep  songwriter  soul  spanish  \
0         0              0  ...    0      0           0     0        0   
1         0              0  ...    0      0           0     0        0   
2         0              0  ...    0      0           0     0        0   
3         0              0  ...    0      0           0     0        0   
4         0              0  ...    0      0           0     0        0   

   swedish  tango  techno  trance  trip-hop  
0        0      0       0   

In [11]:
import numpy as np

def cosine_sim(u1, u2):
    u1 = np.array(u1)
    u2 = np.array(u2)

    dot = np.dot(u1, u2)
    norm_u1 = np.linalg.norm(u1)
    norm_u2 = np.linalg.norm(u2)

    if norm_u1 == 0 or norm_u2 == 0:
        return 0  # tránh chia 0

    return dot / (norm_u1 * norm_u2)

In [12]:
def get_similarity(df2, target_vec, cols_id):
    X = df2.drop(cols_id, axis=1).values
    ids = df2[cols_id].values

    sims = []

    for i, vec in enumerate(X):
        sim = cosine_sim(target_vec, vec)
        sims.append((ids[i], sim))

    # sort giảm dần
    sims.sort(key=lambda x: x[1], reverse=True)

    return sims

In [13]:
# lấy vector của user bất kỳ
target_vec = df2[df2["uid"] == 4].drop("uid", axis=1).values[0]

result_from_user_coll= get_similarity(df2, target_vec, "uid")

for uid, sim in result_from_user_coll[1:5]:
    print(uid, sim)

24 0.6666666666666667
15 0.33333333333333337
22 0.33333333333333337
1 0


In [14]:
query = "select * from case_based;"
mycursor.execute(query)

df_case_based = mycursor.fetchall()

print(df_case_based)
mydb.commit()

[(1, 'power-pop', '1Xyo4u8uXC1ZmMpatF05PJ,6eUKZXaKkcviH0Ku9w2n3V', 0.0), (2, 'ska,dance,soul', '5K4W6rqBFWDnAN6FQUkS6x,6eUKZXaKkcviH0Ku9w2n3V,246dkjvS1zLTtiykXe5h60,3TVXtAsR1Inumwj472S9r4,3Nrfpe0tUJi4K4DXYWgMUX,06HL4z0CvFAxyc27GXpf02', 0.0), (3, 'cantopop', '7dGJo4pcD2V6oG8kP0tJRR,4q3ewBCX7sLwd24euuV69X', 0.0), (4, 'tango,piano,new-age,sleep', '1Xyo4u8uXC1ZmMpatF05PJ,6KImCVD70vtIoJWnq6nGn3,1uNFoZAHBGtllmzznpCI3s,5K4W6rqBFWDnAN6FQUkS6x,06HL4z0CvFAxyc27GXpf02,3Nrfpe0tUJi4K4DXYWgMUX,7dGJo4pcD2V6oG8kP0tJRR', 0.0), (5, 'opera,party,piano,cantopop,samba', '1Xyo4u8uXC1ZmMpatF05PJ,1mcTU81TzQhprhouKaTkpq,3Nrfpe0tUJi4K4DXYWgMUX', 0.0), (6, 'sleep,detroit-techno', '4q3ewBCX7sLwd24euuV69X,1mcTU81TzQhprhouKaTkpq,3TVXtAsR1Inumwj472S9r4,6KImCVD70vtIoJWnq6nGn3,1Xyo4u8uXC1ZmMpatF05PJ,5K4W6rqBFWDnAN6FQUkS6x,06HL4z0CvFAxyc27GXpf02,3Nrfpe0tUJi4K4DXYWgMUX,246dkjvS1zLTtiykXe5h60,6eUKZXaKkcviH0Ku9w2n3V', 0.0), (7, 'singer-songwriter,sertanejo', '1mcTU81TzQhprhouKaTkpq,3Nrfpe0tUJi4K4DXYWgMUX', 0.0), (8, 'deep

In [15]:
import pandas as pd

df_cb = pd.DataFrame(df_case_based, columns=[
    "cbid", "genres", "artist_id", "score"
])
df_cb.head(10)

,cbid,genres,artist_id,score
0,1,power-pop,"1Xyo4u8uXC1ZmMpatF05PJ,6eUKZXaKkcviH0Ku9w2n3V",0.0
1,2,"ska,dance,soul","5K4W6rqBFWDnAN6FQUkS6x,6eUKZXaKkcviH0Ku9w2n3V,...",0.0
2,3,cantopop,"7dGJo4pcD2V6oG8kP0tJRR,4q3ewBCX7sLwd24euuV69X",0.0
3,4,"tango,piano,new-age,sleep","1Xyo4u8uXC1ZmMpatF05PJ,6KImCVD70vtIoJWnq6nGn3,...",0.0
4,5,"opera,party,piano,cantopop,samba","1Xyo4u8uXC1ZmMpatF05PJ,1mcTU81TzQhprhouKaTkpq,...",0.0
5,6,"sleep,detroit-techno","4q3ewBCX7sLwd24euuV69X,1mcTU81TzQhprhouKaTkpq,...",0.0
6,7,"singer-songwriter,sertanejo","1mcTU81TzQhprhouKaTkpq,3Nrfpe0tUJi4K4DXYWgMUX",0.0
7,8,deep-house,"1uNFoZAHBGtllmzznpCI3s,7dGJo4pcD2V6oG8kP0tJRR,...",0.0
8,9,"acoustic,metal,funk,progressive-house,blues,tr...","1Xyo4u8uXC1ZmMpatF05PJ,6eUKZXaKkcviH0Ku9w2n3V,...",0.0
9,10,"trip-hop,afrobeat,dancehall,chicago-house","1Xyo4u8uXC1ZmMpatF05PJ,5K4W6rqBFWDnAN6FQUkS6x,...",0.0


In [16]:
def case_based_genre_matrix(df1):
    # chỉ lấy 2 cột cần
    df = df1[["cbid", "genres"]].copy()

    # xử lý null / rỗng
    df["genres"] = df["genres"].fillna("")
    
    # tách list genre
    df["genres"] = df["genres"].apply(
        lambda x: [g.strip() for g in x.split(",") if g.strip()]
    )

    # explode
    df = df.explode("genres")

    # tạo one-hot
    df = pd.get_dummies(df, columns=["genres"])

    # group lại theo user
    df = df.groupby("cbid").max().reset_index()

    # đảm bảo đủ tất cả genre
    for g in GENRES:
        col = f"genres_{g}"
        if col not in df.columns:
            df[col] = 0

    # rename cho gọn
    df.columns = [c.replace("genres_", "") for c in df.columns]

    # sắp xếp cột
    df = df[["cbid"] + GENRES]

    return df

dfcb = case_based_genre_matrix(df_cb)
dfcb = dfcb.astype(int)
print(dfcb.head())

   cbid  acoustic  afrobeat  alt-rock  ambient  black-metal  blues  breakbeat  \
0     1         0         0         0        0            0      0          0   
1     2         0         0         0        0            0      0          0   
2     3         0         0         0        0            0      0          0   
3     4         0         0         0        0            0      0          0   
4     5         0         0         0        0            0      0          0   

   cantopop  chicago-house  ...  ska  sleep  songwriter  soul  spanish  \
0         0              0  ...    0      0           0     0        0   
1         0              0  ...    1      0           0     1        0   
2         1              0  ...    0      0           0     0        0   
3         0              0  ...    0      1           0     0        0   
4         1              0  ...    0      0           0     0        0   

   swedish  tango  techno  trance  trip-hop  
0        0      0     

In [17]:
# lấy vector của user bất kỳ
target_vec = df2[df2["uid"] == 15].drop("uid", axis=1).values[0]

result_from_case_based = get_similarity(dfcb, target_vec, "cbid")

for cbid, sim in result_from_case_based[:10]:
    print(cbid, sim)

149 0.5773502691896258
208 0.5773502691896258
79 0.5163977794943222
288 0.47140452079103173
20 0.4364357804719848
171 0.4364357804719848
6 0.40824829046386296
144 0.40824829046386296
267 0.40824829046386296
151 0.33333333333333337


In [18]:
# print(result_from_case_based)
dftmp = pd.DataFrame(result_from_case_based, columns=["cbid", "sim"])
dftmp = pd.merge(dftmp, df_cb, on="cbid")
dftmp.head()
# score_artist = []
# cnt = 0
# for i in range(len(dftmp)):
#     arr = dftmp.iloc[i]["artist_id"].split(",")
#     # print(dftmp.iloc[i]["sim"])
#     for j in arr:
#         # print(j)
#         if j not in score_artist:
#             score_artist.append(j)
#             cnt+=1
#         #     score_artist.append((j, float(0)))
#         # score_artist[j][1] += 0.7*float(dftmp.iloc[i]["sim"])

# print(cnt)
from collections import defaultdict

score_artist = defaultdict(float)
k = 0
for _, row in dftmp.iterrows():
    if k == 20:
        break
    sim = float(row["sim"])
    for artist in row["artist_id"].split(","):
        score_artist[artist] += 0.7 * sim

result = list(score_artist.items())

print(result)

[('7dGJo4pcD2V6oG8kP0tJRR', 3.1949385267989596), ('06HL4z0CvFAxyc27GXpf02', 4.589480901332517), ('1mcTU81TzQhprhouKaTkpq', 4.509603853977128), ('6eUKZXaKkcviH0Ku9w2n3V', 4.736647693650507), ('5K4W6rqBFWDnAN6FQUkS6x', 3.700609714298904), ('246dkjvS1zLTtiykXe5h60', 4.3984657628979935), ('1uNFoZAHBGtllmzznpCI3s', 3.173330978823879), ('1Xyo4u8uXC1ZmMpatF05PJ', 4.239682635937966), ('3Nrfpe0tUJi4K4DXYWgMUX', 3.410852308514666), ('3TVXtAsR1Inumwj472S9r4', 4.855439603816389), ('4q3ewBCX7sLwd24euuV69X', 4.308370115390316), ('6KImCVD70vtIoJWnq6nGn3', 4.46272476694318)]


In [20]:
dftmp1 = pd.DataFrame(result_from_user_coll, columns=["uid", "sim"])
dftmp1 = pd.merge(dftmp1, df1, on="uid")
dftmp1.head()

,uid,sim,favorite_genre,artist_id
0,4,1.000000,"tango, techno, trance","1Cs0zKBU1kc0i8ypK3B9ai, 2R21vXR83lH98kGeO99Y66..."
1,24,0.666667,"german, techno, trance","0Y5tJX1MQlPlqiwlOH1tJY, 2LRoIwlKmHjgvigdNGBHNo..."
2,15,0.333333,"techno, minimal-techno, detroit-techno","0du5cEVh5yTK9QJze8zA0C, 1McMsnEElThX1knmY4oliG..."
3,22,0.333333,"salsa, samba, tango","00FQb4jTyendYWaN8pK0wa, 3Nrfpe0tUJi4K4DXYWgMUX..."
4,1,0.000000,NaN,"0EmeFodog0BfCgMzAIvKQp, 3Nrfpe0tUJi4K4DXYWgMUX..."


In [21]:
from collections import defaultdict

# score_artist = defaultdict(float)
k = 0
for _, row in dftmp1.iterrows():
    if k == 20:
        break
    sim = float(row["sim"])
    if sim == 1:
        continue
    for artist in row["artist_id"].split(","):
        score_artist[artist] += 0.3 * sim

result = list(score_artist.items())


In [23]:
result = sorted(score_artist.items(), key=lambda x: x[1], reverse=True)
print(result)

[('3TVXtAsR1Inumwj472S9r4', 4.855439603816389), ('6eUKZXaKkcviH0Ku9w2n3V', 4.736647693650507), ('06HL4z0CvFAxyc27GXpf02', 4.589480901332517), ('1mcTU81TzQhprhouKaTkpq', 4.509603853977128), ('6KImCVD70vtIoJWnq6nGn3', 4.46272476694318), ('246dkjvS1zLTtiykXe5h60', 4.3984657628979935), ('4q3ewBCX7sLwd24euuV69X', 4.308370115390316), ('1Xyo4u8uXC1ZmMpatF05PJ', 4.239682635937966), ('5K4W6rqBFWDnAN6FQUkS6x', 3.700609714298904), ('3Nrfpe0tUJi4K4DXYWgMUX', 3.410852308514666), ('7dGJo4pcD2V6oG8kP0tJRR', 3.1949385267989596), ('1uNFoZAHBGtllmzznpCI3s', 3.173330978823879), (' 6l3HvQ5sa6mXTsMTB19rO5', 0.7), (' 2R21vXR83lH98kGeO99Y66', 0.5), (' 6M2wZ9GZgrQXHCFfjv46we', 0.5), ('1Cs0zKBU1kc0i8ypK3B9ai', 0.30000000000000004), (' 3MZsBdqDrRTJihTHQrO6Dq', 0.30000000000000004), (' 3TVXtAsR1Inumwj472S9r4', 0.30000000000000004), (' 4VMYDCV2IEDYJArk749S6m', 0.30000000000000004), (' 7Ln80lUS6He07XvHI8qqHH', 0.30000000000000004), (' 53XhwfbYqKCa1cC15pYq2q', 0.30000000000000004), ('0Y5tJX1MQlPlqiwlOH1tJY', 0.2), 